# Туториал: Local-Disk streaming (`streaming.mode=local_disk`)

Цель: запуск с финальными chunk-файлами на локальном диске.

Подходит для:
- single-node multi-GPU
- DDP c chunk sharding
- устойчивого producer/consumer буфера


In [ ]:
from __future__ import annotations

import sys
import platform
from pathlib import Path
from collections import Counter

import torch
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.shared.collector_service import CollectorService
from dataset.shared.shared_dataset import SharedModelDataset

def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        return compose(config_name=cfg_path.stem, overrides=overrides or [])

print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device count: {torch.cuda.device_count()}')


## Подготовка конфигурации

Используем профиль `streaming=gpu_parallel_streaming`.

Если у вас 2 GPU, рекомендуемый вариант:
- `train.device=cuda:0`
- `collector.device=cuda:1`

Если GPU одна — можно временно поставить `collector.device=null` для interleaved.


In [ ]:
if torch.cuda.device_count() >= 2:
    TRAIN_DEVICE = "cuda:0"
    COLLECTOR_DEVICE = "cuda:1"
else:
    TRAIN_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    COLLECTOR_DEVICE = "null"

overrides = [
    "streaming=gpu_parallel_streaming",
    "data.path=./data",
    f"train.device={TRAIN_DEVICE}",
    f"collector.device={COLLECTOR_DEVICE}",
    "collector.mode=auto",
    "data.enabled_datasets=[coco2017,scene_parse_150]",
    "data.dataset_overrides.coco2017.models=[clip_vit_b32]",
    "data.dataset_overrides.scene_parse_150.models=[dinov2_base]",
    "streaming.local_disk.max_ready_chunks=40",
    "streaming.local_disk.low_watermark_chunks=20",
    "streaming.chunk_size_samples=64",
]

cfg = load_hydra_cfg(overrides=overrides)
print('streaming mode =', cfg.streaming.mode)
print(OmegaConf.to_yaml(cfg.streaming, resolve=True))
print(OmegaConf.to_yaml(cfg.collector, resolve=True))


In [ ]:
collector = CollectorService(cfg)
dataset = SharedModelDataset(collector)
collector.start()

model_counts = Counter()
consumed = 0

try:
    for step in range(120):
        if not collector.is_async_mode:
            dataset.maybe_collect(step)

        sample = dataset.try_next_sample()
        if sample is None:
            continue

        consumed += 1
        model_counts[sample.model_name] += 1

    print('consumed:', consumed)
    print('ready_chunk_metric:', dataset.cache_size())
    print('model_counts:', dict(model_counts))
    print('collector stats:', collector.stats())
finally:
    dataset.close()
    collector.shutdown()


## Проверка dedicated GPU поведения

В async профиле по умолчанию включено:
- `collector.pin_gpu=true`
- `collector.release_device_on_unload=false`
- `collector.empty_cuda_cache_on_unload=false`

То есть collector на выделенной GPU не пытается агрессивно освобождать память под train/VAE.
